##### ***词的相似性和类比任务***
###### 在前文中，我们使用一个小小的数据集实现了word2vec，并使用它为一个输入词寻找语义相似的词，事实上，我们在大型语料库预先训练的词向量可以应用于下游的自然语言处理任务，而为了直观展示大型语料库中预训练词向量的语义，让我们将预训练词向量应用到词的相似性和类别任务中。

In [1]:
import os
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# 以下列出维度为50、100和300的预训练GloVe嵌入。预训练的fastText嵌入有多种语言。这里我们使用可以从fastText网站下载300维度的英文版本（“wiki.en”）。
#@save
d2l.DATA_HUB['glove.6b.50d'] = (d2l.DATA_URL + 'glove.6B.50d.zip',
                                '0b8703943ccdb6eb788e6f091b8946e82231bc4d')

#@save
d2l.DATA_HUB['glove.6b.100d'] = (d2l.DATA_URL + 'glove.6B.100d.zip',
                                 'cd43bfb07e44e6f27cbcc7bc9ae3d80284fdaf5a')

#@save
d2l.DATA_HUB['glove.42b.300d'] = (d2l.DATA_URL + 'glove.42B.300d.zip',
                                  'b5116e234e9eb9076672cfeabf5469f3eec904fa')

#@save
d2l.DATA_HUB['wiki.en'] = (d2l.DATA_URL + 'wiki.en.zip',
                           'c1816da3821ae9f43899be655002f6c723e91b88')

In [ ]:
class TokenEmbedding:
    """GloVe嵌入"""
    def __init__(self, embedding_name):
        # 获取GloVe嵌入的索引到词的映射，以及所有词向量
        self.idx_to_token, self.idx_to_vec = self._load_embedding(embedding_name)
        self.unknown_idx = 0 # 未知词的索引为0
        # 构建词到索引的映射
        self.token_to_idx = {token: idx for idx, token in
                             enumerate(self.idx_to_token)}
    
    def _load_embedding(self, embedding_name):
        # 初始化两个列表
        idx_to_token, idx_to_vec = ['<unk>'], []
        data_dir = d2l.download_extract(embedding_name)
        with open(os.path.join(data_dir, 'vec.txt'), 'r') as f:
            # vec.txt文件中的数据格式为：每行第一个元素是词，后面是该词的向量表示
            for line in f:
                # 去掉行末的换行符，并以空格分割成列表
                elems = line.rstrip().split(' ')
                token, elems = elems[0], [float(elem) for elem in elems[1:]]
                if len(elems) > 1:  # 跳过可能的标题行
                    idx_to_token.append(token)
                    idx_to_vec.append(elems)
        # 在第一行添加一个全零的向量，表示未知词的向量
        idx_to_vec = [[0] * len(idx_to_vec[0])] + idx_to_vec
        return idx_to_token, torch.tensor(idx_to_vec)

    def __getitem__(self, tokens):
        indices = [self.token_to_idx.get(token, self.unknown_idx)
                   for token in tokens]
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs

    def __len__(self):
        return len(self.idx_to_token)

In [5]:
glove_6b50d = TokenEmbedding('glove.6b.50d')

In [8]:
print(len(glove_6b50d))
glove_6b50d.token_to_idx['beautiful'], glove_6b50d.idx_to_token[3367]

400001


(3367, 'beautiful')

###### 应用预训练的词向量到词相似度任务和类比任务中，在词相似度任务重，我们使用k紧邻(k-Nearest Neighbor, KNN)算法来找到与给定词向量最相似的k个词。一般使用余弦相似度来计算:
$$cosine\_sim(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$$
###### 词类比任务例如“man” : “woman” :: “son” : “daughter”是一个词的类比。 “man”是对“woman”的类比，“son”是对“daughter”的类比。具体来说，词类比任务可以定义为： 对于单词类比“a : b :: c : d”，给出前三个词“a”、“b”、“c”，要求找到一个词“d”，使得“a : b :: c : d”是一个词的类比。用$vec(w)$表示词$w$的向量，为了完成这个类比，我们将找到一个词，其向量与$vec(c) + vec(b) - vec(a)$的结果最相似。

In [9]:
# 词相似度任务
def knn(W, x, k):
    # torch.mv用于计算矩阵和向量的乘积。W是一个矩阵，x是一个向量。torch.mv(W, x)将返回一个新的向量，其中每个元素是W的行与x的点积。
    cos = torch.mv(W, x.reshape(-1,)) / (
        torch.sqrt(torch.sum(W * W, axis=1) + 1e-9) *
        torch.sqrt((x * x).sum()))
    _, topk = torch.topk(cos, k=k)
    return topk, [cos[int(i)] for i in topk]

In [ ]:
def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):  # 排除输入词
        print(f'{embed.idx_to_token[int(i)]}：cosine相似度={float(c):.3f}')

In [17]:
get_similar_tokens('chip', 3, glove_6b50d), print('\n')
get_similar_tokens('baby', 3, glove_6b50d), print('\n')
get_similar_tokens('beautiful', 3, glove_6b50d)

chips：cosine相似度=0.856
intel：cosine相似度=0.749
electronics：cosine相似度=0.749


babies：cosine相似度=0.839
boy：cosine相似度=0.800
girl：cosine相似度=0.792


lovely：cosine相似度=0.921
gorgeous：cosine相似度=0.893
wonderful：cosine相似度=0.830


In [18]:
def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] - vecs[0] + vecs[2]
    topk, cos = knn(embed.idx_to_vec, x, 1)
    return embed.idx_to_token[int(topk[0])] 

In [20]:
print(get_analogy('man', 'woman', 'son', glove_6b50d)), print('\n')
print(get_analogy('beijing', 'china', 'tokyo', glove_6b50d)), print('\n')
print(get_analogy('bad', 'worst', 'big', glove_6b50d)), print('\n')
print(get_analogy('do', 'did', 'go', glove_6b50d))

daughter


japan


biggest


went
